In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

print("All libraries are working in Jupyter Notebook!")

All libraries are working in Jupyter Notebook!


In [29]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in current folder:")
print(os.listdir())

Current folder:
c:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\Notebooks

Files in current folder:
['Airtel_Netflix_Amazon_Workflow.ipynb', 'Amazon_Proxy_Churn_Workflow.ipynb', 'Customer_Churn_Project.ipynb']


In [30]:
import pandas as pd

df_jio = pd.read_csv("../Datasets/Jio/Jio_Customer_Data.csv")

print(df_jio.shape)
df_jio.head()

(6418, 32)


,Customer_ID,Gender,Age,Married,State,Number_of_Referrals,Tenure_in_Months,Value_Deal,Phone_Service,Multiple_Lines,...,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Churn_Category,Churn_Reason
0,19877-DEL,Male,35,No,Delhi,7,27,NaN,Yes,No,...,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,58353-MAH,Female,45,Yes,Maharashtra,14,13,NaN,Yes,Yes,...,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,25063-WES,Male,51,No,West Bengal,4,35,Deal 5,Yes,No,...,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,59787-KAR,Male,79,No,Karnataka,3,21,Deal 4,Yes,No,...,Bank Withdrawal,98.0,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,28544-TAM,Female,80,No,Tamil Nadu,3,8,NaN,Yes,No,...,Credit Card,83.9,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [31]:
df_jio_clean = df_jio.copy()

print("Original dataset:", df_jio.shape)
print("Working copy:", df_jio_clean.shape)


Original dataset: (6418, 32)
Working copy: (6418, 32)


In [32]:
df_jio_clean.isnull().sum()

Customer_ID                       0
Gender                            0
Age                               0
Married                           0
State                             0
Number_of_Referrals               0
Tenure_in_Months                  0
Value_Deal                     3548
Phone_Service                     0
Multiple_Lines                  622
Internet_Service                  0
Internet_Type                  1390
Online_Security                1390
Online_Backup                  1390
Device_Protection_Plan         1390
Premium_Support                1390
Streaming_TV                   1390
Streaming_Movies               1390
Streaming_Music                1390
Unlimited_Data                 1390
Contract                          0
Paperless_Billing                 0
Payment_Method                    0
Monthly_Charge                    0
Total_Charges                     0
Total_Refunds                     0
Total_Extra_Data_Charges          0
Total_Long_Distance_Charges 

In [33]:
df_jio_clean.groupby("Customer_Status")[
    ["Churn_Category", "Churn_Reason"]
].apply(lambda x: x.isnull().sum())

,Churn_Category,Churn_Reason
Customer_Status,,
Churned,0,0
Joined,411,411
Stayed,4275,4275


In [34]:
df_jio_clean["Churn_Category"] = (
    df_jio_clean["Churn_Category"].fillna("Not Applicable")
)

df_jio_clean["Churn_Reason"] = (
    df_jio_clean["Churn_Reason"].fillna("Not Applicable")
)

print(df_jio_clean[["Churn_Category", "Churn_Reason"]].isnull().sum())

Churn_Category    0
Churn_Reason      0
dtype: int64


In [35]:
df_jio_clean.isnull().sum()[df_jio_clean.isnull().sum() > 0]

Value_Deal                3548
Multiple_Lines             622
Internet_Type             1390
Online_Security           1390
Online_Backup             1390
Device_Protection_Plan    1390
Premium_Support           1390
Streaming_TV              1390
Streaming_Movies          1390
Streaming_Music           1390
Unlimited_Data            1390
dtype: int64

In [36]:
# Fill missing categorical values
categorical_cols = df_jio_clean.select_dtypes(
    include="object"
).columns

for col in categorical_cols:
    if df_jio_clean[col].isnull().sum() > 0:
        df_jio_clean[col] = df_jio_clean[col].fillna("Unknown")

# Check remaining missing values
print(df_jio_clean.isnull().sum().sum())

0


C:\Users\mudda\AppData\Local\Temp\ipykernel_14784\496059067.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_jio_clean.select_dtypes(


In [37]:
print("Duplicate rows:", df_jio_clean.duplicated().sum())

Duplicate rows: 0


In [38]:
df_jio_clean["Customer_Status"].value_counts()

Customer_Status
Stayed     4275
Churned    1732
Joined      411
Name: count, dtype: int64

In [39]:
# Keep only customers who stayed or churned
df_jio_model = df_jio_clean[
    df_jio_clean["Customer_Status"].isin(["Stayed", "Churned"])
].copy()

# Create binary target: 1 = Churned, 0 = Stayed
df_jio_model["Churn"] = (
    df_jio_model["Customer_Status"] == "Churned"
).astype(int)

print(df_jio_model["Churn"].value_counts())
print("Dataset shape:", df_jio_model.shape)

Churn
0    4275
1    1732
Name: count, dtype: int64
Dataset shape: (6007, 33)


In [40]:
print(df_jio_model["Churn"].value_counts())

print("\nChurn percentage:")
print(df_jio_model["Churn"].value_counts(normalize=True) * 100)

Churn
0    4275
1    1732
Name: count, dtype: int64

Churn percentage:
Churn
0    71.166972
1    28.833028
Name: proportion, dtype: float64


In [41]:
# Columns we should not use as prediction features
cols_to_drop = [
    "Customer_ID",
    "Customer_Status",
    "Churn_Category",
    "Churn_Reason"
]

df_jio_model = df_jio_model.drop(columns=cols_to_drop)

print("Dataset shape:", df_jio_model.shape)
print(df_jio_model.columns.tolist())

Dataset shape: (6007, 29)
['Gender', 'Age', 'Married', 'State', 'Number_of_Referrals', 'Tenure_in_Months', 'Value_Deal', 'Phone_Service', 'Multiple_Lines', 'Internet_Service', 'Internet_Type', 'Online_Security', 'Online_Backup', 'Device_Protection_Plan', 'Premium_Support', 'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data', 'Contract', 'Paperless_Billing', 'Payment_Method', 'Monthly_Charge', 'Total_Charges', 'Total_Refunds', 'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges', 'Total_Revenue', 'Churn']


In [42]:
df_jio_model.info()

<class 'pandas.DataFrame'>
Index: 6007 entries, 0 to 6417
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Gender                       6007 non-null   str    
 1   Age                          6007 non-null   int64  
 2   Married                      6007 non-null   str    
 3   State                        6007 non-null   str    
 4   Number_of_Referrals          6007 non-null   int64  
 5   Tenure_in_Months             6007 non-null   int64  
 6   Value_Deal                   6007 non-null   str    
 7   Phone_Service                6007 non-null   str    
 8   Multiple_Lines               6007 non-null   str    
 9   Internet_Service             6007 non-null   str    
 10  Internet_Type                6007 non-null   str    
 11  Online_Security              6007 non-null   str    
 12  Online_Backup                6007 non-null   str    
 13  Device_Protection_Plan       6007 

In [43]:
categorical_cols = df_jio_model.select_dtypes(
    include=["str", "object"]
).columns.tolist()

print("Number of categorical columns:", len(categorical_cols))
print(categorical_cols)

Number of categorical columns: 19
['Gender', 'Married', 'State', 'Value_Deal', 'Phone_Service', 'Multiple_Lines', 'Internet_Service', 'Internet_Type', 'Online_Security', 'Online_Backup', 'Device_Protection_Plan', 'Premium_Support', 'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data', 'Contract', 'Paperless_Billing', 'Payment_Method']


In [44]:
df_jio_encoded = pd.get_dummies(
    df_jio_model,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

print("Shape before encoding:", df_jio_model.shape)
print("Shape after encoding:", df_jio_encoded.shape)

df_jio_encoded.head()

Shape before encoding: (6007, 29)
Shape after encoding: (6007, 66)


,Age,Number_of_Referrals,Tenure_in_Months,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Churn,...,Streaming_Movies_Yes,Streaming_Music_Unknown,Streaming_Music_Yes,Unlimited_Data_Unknown,Unlimited_Data_Yes,Contract_One Year,Contract_Two Year,Paperless_Billing_Yes,Payment_Method_Credit Card,Payment_Method_Mailed Check
0,35,7,27,65.6,593.30,0.00,0,381.51,974.81,0,...,0,0,0,0,1,1,0,1,1,0
1,45,14,13,-4.0,542.40,38.33,10,96.21,610.28,0,...,1,0,1,0,0,0,0,0,1,0
2,51,4,35,73.9,280.85,0.00,0,134.60,415.45,1,...,0,0,0,0,1,0,0,1,0,0
3,79,3,21,98.0,1237.85,0.00,0,361.66,1599.51,1,...,1,0,0,0,1,0,0,1,0,0
4,80,3,8,83.9,267.40,0.00,0,22.14,289.54,1,...,0,0,0,0,1,0,0,1,1,0


In [45]:
print("Negative monthly charges:",
      (df_jio_encoded["Monthly_Charge"] < 0).sum())

print("Negative total charges:",
      (df_jio_encoded["Total_Charges"] < 0).sum())

print("Negative tenure:",
      (df_jio_encoded["Tenure_in_Months"] < 0).sum())

Negative monthly charges: 101
Negative total charges: 0
Negative tenure: 0


In [46]:
X = df_jio_encoded.drop(columns=["Churn"])
y = df_jio_encoded["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (6007, 65)
Target shape: (6007,)


In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

Training features: (4805, 65)
Testing features: (1202, 65)
Training targets: (4805,)
Testing targets: (1202,)


In [48]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training target distribution:
Churn
0    3420
1    1385
Name: count, dtype: int64

Testing target distribution:
Churn
0    855
1    347
Name: count, dtype: int64


In [49]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Separate features and target
X = df_jio_model.drop(columns=["Churn"])
y = df_jio_model["Churn"]

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["str", "object"]).columns
numerical_cols = X.select_dtypes(include=["number"]).columns

# Preprocess numerical and categorical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Split data before fitting preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build and train the model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.831946755407654

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.90      0.88       855
           1       0.73      0.65      0.69       347

    accuracy                           0.83      1202
   macro avg       0.80      0.78      0.79      1202
weighted avg       0.83      0.83      0.83      1202



In [50]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Separate features and target
X = df_jio_model.drop(columns=["Churn"])
y = df_jio_model["Churn"]

# Identify column types
categorical_cols = X.select_dtypes(
    include=["str", "object"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Preprocess numerical data
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical data
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Split before fitting preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Build and train the model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

# Evaluate the model
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.831946755407654

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.90      0.88       855
           1       0.73      0.65      0.69       347

    accuracy                           0.83      1202
   macro avg       0.80      0.78      0.79      1202
weighted avg       0.83      0.83      0.83      1202



In [51]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Random Forest using the same preprocessing pipeline
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)

# Predict on test data
rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

Random Forest Accuracy: 0.8452579034941764

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.89      0.89       855
           1       0.73      0.73      0.73       347

    accuracy                           0.85      1202
   macro avg       0.81      0.81      0.81      1202
weighted avg       0.84      0.85      0.85      1202



In [52]:
import joblib
from pathlib import Path

models_dir = Path("../models")
reports_dir = Path("../reports")
model_path = models_dir / "jio_churn_model.pkl"
report_path = reports_dir / "jio_model_comparison.csv"
existing_outputs = [str(path.resolve()) for path in (model_path, report_path) if path.exists()]
if existing_outputs:
    raise FileExistsError(
        "Refusing to overwrite existing Jio outputs. No files were written. "
        "Choose new output paths or explicitly preserve/archive the existing outputs first:\n"
        + "\n".join(existing_outputs)
    )

models_dir.mkdir(parents=True, exist_ok=True)
with model_path.open("xb") as model_file:
    joblib.dump(rf_model, model_file)

print("Random Forest model saved successfully!")
print("Location:", model_path)

FileExistsError: Refusing to overwrite existing Jio outputs. No files were written. Choose new output paths or explicitly preserve/archive the existing outputs first:
C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\models\jio_churn_model.pkl
C:\Users\mudda\OneDrive - BENNETT UNIVERSITY\Customer Churn Preddiction & Analytics Dashboard\reports\jio_model_comparison.csv

In [ ]:
import joblib

loaded_model = joblib.load("../models/jio_churn_model.pkl")

print("Model loaded successfully!")
print("Test predictions:", loaded_model.predict(X_test.head(5)))

Model loaded successfully!
Test predictions: [0 0 1 1 1]


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Churn Precision": precision_score(y_test, y_pred),
        "Churn Recall": recall_score(y_test, y_pred),
        "Churn F1-score": f1_score(y_test, y_pred)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Churn Precision": precision_score(y_test, rf_pred),
        "Churn Recall": recall_score(y_test, rf_pred),
        "Churn F1-score": f1_score(y_test, rf_pred)
    }
])

results

,Model,Accuracy,Churn Precision,Churn Recall,Churn F1-score
0,Logistic Regression,0.831947,0.734628,0.654179,0.692073
1,Random Forest,0.845258,0.733333,0.729107,0.731214


In [ ]:
from pathlib import Path

report_path = Path("../reports/jio_model_comparison.csv")
if report_path.exists():
    raise FileExistsError(
        "Refusing to overwrite the existing Jio comparison report. No file was written. "
        "Choose a new report path or explicitly preserve/archive the existing report first: "
        + str(report_path.resolve())
    )
results.to_csv(report_path, index=False, mode="x")
print("Model comparison saved successfully!")

Model comparison saved successfully!


In [53]:
import joblib
from pathlib import Path

model_path = Path("../models/jio_churn_model.pkl")
loaded_model = joblib.load(model_path)

print("Jio model loaded successfully!")

Jio model loaded successfully!


In [54]:
predictions = loaded_model.predict(X_test)

churn_customers = X_test[predictions == 1]

if len(churn_customers) > 0:
    print("Enter these values in your form:")
    display(churn_customers.iloc[[0]])
    print("Expected output: Customer is not likely to stay.")
else:
    print("No churn predictions found.")

Enter these values in your form:


,Gender,Age,Married,State,Number_of_Referrals,Tenure_in_Months,Value_Deal,Phone_Service,Multiple_Lines,Internet_Service,...,Unlimited_Data,Contract,Paperless_Billing,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue
2535,Female,66,Yes,Gujarat,7,12,Deal 5,Yes,No,Yes,...,Yes,Month-to-Month,No,Credit Card,69.7,516.15,0.0,0,36.19,552.34


Expected output: Customer is not likely to stay.


In [55]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Create input fields from the Jio training data
fields = {}

for col in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[col]):
        value = float(X_train[col].median())
        fields[col] = widgets.FloatText(
            value=value,
            description=col,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="450px")
        )
    else:
        options = sorted(X_train[col].dropna().unique().tolist())
        fields[col] = widgets.Dropdown(
            options=options,
            description=col,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="450px")
        )

button = widgets.Button(
    description="Predict Churn",
    button_style="success"
)
output = widgets.Output()

def predict_churn(b):
    with output:
        output.clear_output()

        customer = pd.DataFrame([{
            col: field.value for col, field in fields.items()
        }], columns=X_train.columns)

        prediction = loaded_model.predict(customer)[0]

        print("CUSTOMER INPUT:")
        display(customer)

        print("\nPREDICTION OUTPUT:")
        if prediction == 1:
            print("Customer is likely to churn.")
        else:
            print("Customer is likely to stay.")

button.on_click(predict_churn)

display(widgets.VBox(list(fields.values()) + [button, output]))

In [56]:
predictions = loaded_model.predict(X_test)

churn_rows = X_test.loc[predictions == 1]

if not churn_rows.empty:
    sample = churn_rows.iloc[[0]]

    print("Copy these exact input values:")
    display(sample)

    print("Model prediction:",
          loaded_model.predict(sample)[0])
else:
    print("No churn predictions found.")

Copy these exact input values:


,Gender,Age,Married,State,Number_of_Referrals,Tenure_in_Months,Value_Deal,Phone_Service,Multiple_Lines,Internet_Service,...,Unlimited_Data,Contract,Paperless_Billing,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue
2535,Female,66,Yes,Gujarat,7,12,Deal 5,Yes,No,Yes,...,Yes,Month-to-Month,No,Credit Card,69.7,516.15,0.0,0,36.19,552.34


Model prediction: 1
